# Stellar Classification: Model Tuning & Ensemble Optimization

This notebook executes probability-blending weight optimization across LightGBM, XGBoost, and CatBoost out-of-fold (OOF) predictions. By finding the optimal blend weights, we maximize the cross-validation Balanced Accuracy score and generate the final predictions.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score
import os

# Target mapping variables
TARGET_MAP = {'GALAXY': 0, 'QSO': 1, 'STAR': 2}
INV_TARGET_MAP = {0: 'GALAXY', 1: 'QSO', 2: 'STAR'}

## 1. Directory Configurations & Dynamic Outputs

We configure dataset directories, prediction probability sources, and target output paths, ensuring portability across local and Kaggle environments.

In [ ]:
# Dynamic path resolution for Local vs Kaggle environment
if os.path.exists('/kaggle/input/competitions/playground-series-s6e6'):
    DATA_DIR = '/kaggle/input/competitions/playground-series-s6e6'
    OUTPUT_DIR = '.'
    
    # Search for notebook 2 prediction artifacts on Kaggle
    potential_dirs = [
        './predictions',
        '/kaggle/input/notebooks/tuannm3812/stellar-classification-baseline-modeling/predictions',
        '/kaggle/input/notebooks/tuannm3812/stellar-classification-baseline-modeling',
        '/kaggle/input/stellar-classification-baseline-modeling/predictions',
        '/kaggle/input/stellar-classification-baseline-modeling'
    ]
    PRED_DIR = './predictions'
    for p_dir in potential_dirs:
        if os.path.exists(p_dir) and os.path.exists(os.path.join(p_dir, 'lgb_oof.npy')):
            PRED_DIR = p_dir
            break
else:
    DATA_DIR = '../data'
    PRED_DIR = '../predictions'
    OUTPUT_DIR = '..'

print(f"Data Directory: {DATA_DIR}")
print(f"Predictions Directory: {PRED_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")

## 2. Loading Probability Predictions & Target Labels

We load the ground-truth training classes alongside the saved out-of-fold validation and test prediction probability matrices. Blending prediction probabilities (confidence scores) rather than discrete labels maintains calibration and improves class boundary separation.

In [ ]:
# Load ground truth class labels
train_path = os.path.join(DATA_DIR, 'train.csv')
train = pd.read_csv(train_path)
y = train['class'].map(TARGET_MAP).values

# Load OOF prediction probabilities
lgb_oof = np.load(os.path.join(PRED_DIR, 'lgb_oof.npy'))
xgb_oof = np.load(os.path.join(PRED_DIR, 'xgb_oof.npy'))
cat_oof = np.load(os.path.join(PRED_DIR, 'cat_oof.npy'))

# Load test prediction probabilities
lgb_test = np.load(os.path.join(PRED_DIR, 'lgb_test.npy'))
xgb_test = np.load(os.path.join(PRED_DIR, 'xgb_test.npy'))
cat_test = np.load(os.path.join(PRED_DIR, 'cat_test.npy'))

print(f"Loaded shapes - y: {y.shape}, LGB OOF: {lgb_oof.shape}, XGB OOF: {xgb_oof.shape}, Cat OOF: {cat_oof.shape}")

## 3. Individual Baseline Performance Review

We verify the baseline Balanced Accuracy scores of the individual models before ensembling. This establishes the baseline performance threshold that our ensemble optimization must exceed.

In [ ]:
lgb_score = balanced_accuracy_score(y, np.argmax(lgb_oof, axis=1))
xgb_score = balanced_accuracy_score(y, np.argmax(xgb_oof, axis=1))
cat_score = balanced_accuracy_score(y, np.argmax(cat_oof, axis=1))

print("--- Individual Model Scores (Balanced Accuracy) ---")
print(f"LightGBM: {lgb_score:.5f}")
print(f"XGBoost:  {xgb_score:.5f}")
print(f"CatBoost: {cat_score:.5f}")

## 4. Leakage-Free Grid Search for Probability Blending

### Ensembling via Probability Blending
We combine individual model outputs using a weighted average of their probability predictions:

$$P_{\text{blend}} = w_{\text{LGB}} P_{\text{LGB}} + w_{\text{XGB}} P_{\text{XGB}} + w_{\text{Cat}} P_{\text{Cat}}$$
$$\text{subject to } w_{\text{LGB}} + w_{\text{XGB}} + w_{\text{Cat}} = 1.0 \quad \text{and} \quad w_m \ge 0$$

Blending diverse tree split models (leaf-wise, level-wise, and symmetric) reduces generalization variance by smoothing out individual framework biases.

### Prevention of Data Leakage
To prevent overfitting, the search for optimal weights is performed strictly on **Out-of-Fold (OOF)** predictions. OOF predictions represent validation folds that were unseen by their respective models during training. Finding weights that maximize OOF Balanced Accuracy ensures that the resulting ensemble weights generalize robustly to the unseen test set.

In [ ]:
best_score = 0
best_weights = None

# Search across grid weights with a step size of 0.05
for w_lgb in np.linspace(0, 1, 21):
    for w_xgb in np.linspace(0, 1 - w_lgb, 21):
        w_cat = 1.0 - w_lgb - w_xgb
        if w_cat < -1e-5: 
            continue
        w_cat = max(0.0, w_cat)
        
        # Calculate probability blend
        blend_oof = w_lgb * lgb_oof + w_xgb * xgb_oof + w_cat * cat_oof
        blend_score = balanced_accuracy_score(y, np.argmax(blend_oof, axis=1))
        
        if blend_score > best_score:
            best_score = blend_score
            best_weights = (w_lgb, w_xgb, w_cat)

w_lgb, w_xgb, w_cat = best_weights
print("--- Best Ensemble Result ---")
print(f"Optimal Balanced Accuracy: {best_score:.5f}")
print(f"Optimal Weights -> LightGBM: {w_lgb:.2f}, XGBoost: {w_xgb:.2f}, CatBoost: {w_cat:.2f}")

## 5. Final Blend Inference & Submission Formulation

Using the optimized blending weights, we blend the test set probability matrices, assign the final class codes, and apply the inverse target mapping to output text labels (`GALAXY`, `QSO`, `STAR`) for the final submission.

In [ ]:
# Applying optimized weights to the test predictions
final_test_prob = w_lgb * lgb_test + w_xgb * xgb_test + w_cat * cat_test
final_preds = np.argmax(final_test_prob, axis=1)

# Create submission DataFrame
sub_sample_path = os.path.join(DATA_DIR, 'sample_submission.csv')
submission = pd.read_csv(sub_sample_path)
submission['class'] = [INV_TARGET_MAP[p] for p in final_preds]

# Save to CSV
sub_out_path = os.path.join(OUTPUT_DIR, 'submission.csv')
submission.to_csv(sub_out_path, index=False)
print(f"Submission file created successfully at {sub_out_path}!")
print(submission['class'].value_counts())